# Day 3 模块 6：预测与简单评估

把模型真正用起来：给定一天的条件，预测营收。


In [ ]:
from pathlib import Path
import pandas as pd

candidate_paths = [
    Path('day3_cafe_sales.csv'),
    Path('day3') / 'day3_cafe_sales.csv',
    Path('教学课程') / 'day3' / 'day3_cafe_sales.csv',
]
for path in candidate_paths:
    if path.exists():
        csv_path = path
        break
else:
    raise FileNotFoundError('未找到 day3_cafe_sales.csv')

df = pd.read_csv(csv_path)
print(csv_path.resolve())
print('shape:', df.shape)
df.head()


In [ ]:
# 准备特征和目标（沿用 Day 2 的思路）
feature_cols = ['price', 'promotion', 'is_weekend', 'temp', 'quality', 'competitors', 'holiday', 'location']
# 天气变成数字分：晴最好，大雨最差
weather_score_map = {'晴': 1.0, '多云': 0.8, '阴': 0.6, '小雨': 0.4, '大雨': 0.3}
df = df.copy()
df['weather_score'] = df['weather_label'].map(weather_score_map)

X = df[feature_cols + ['weather_score']].copy()
y = df['sales'].copy()

print('特征列:', list(X.columns))
print('样本数:', len(X))
X.head()


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print('训练集行数:', len(X_train))
print('测试集行数:', len(X_test))


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=8,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print('测试集 R²:', round(r2_score(y_test, y_pred), 3))
print('测试集 MAE:', round(mean_absolute_error(y_test, y_pred), 1))
print('（MAE 可以粗懂成：平均大概偏多少钱）')


## 1. R² 先记直觉

前面说过：训练时会看预测和真实差多少（损失），训完后要评估好不好。

今天用两个简单读数：

- **R²**：越接近 1，通常表示模型解释波动越多
- **MAE**：平均大概偏多少钱（和“损失大小”是同一类感觉）

- R² 接近 0：几乎没比“瞎猜平均”好多少
- 具体公式后面可以再加深

今天会读“测试集 R² 是多少”就够。


## 2. 构造“新的一天”来预测

假设明天：

- 价格 28
- 有促销
- 周末
- 气温 22
- 品质 8
- 竞争对手 2
- 不是节日
- 位置分 7
- 天气评分 0.8（多云附近）


In [ ]:
import numpy as np

new_day = pd.DataFrame([{
    'price': 28,
    'promotion': 1,
    'is_weekend': 1,
    'temp': 22,
    'quality': 8,
    'competitors': 2,
    'holiday': 0,
    'location': 7,
    'weather_score': 0.8,
}])

pred = rf.predict(new_day)[0]
print('预测营收约:', round(pred, 1))


## 3. 改一个条件，看预测怎么变

例如：把促销关掉，或改成大雨（weather_score=0.3）。


In [ ]:
new_day2 = new_day.copy()
new_day2['promotion'] = 0
print('无促销预测:', round(rf.predict(new_day2)[0], 1))

new_day3 = new_day.copy()
new_day3['weather_score'] = 0.3
print('大雨附近预测:', round(rf.predict(new_day3)[0], 1))


## 4. 模块小结

今天你已经走完最小建模闭环：

```text
读数据 → 准备 X/y → 划分 → 训练随机森林 → 看重要性 → 预测 → 看 R²
```

后面 Day 4 会比较更多算法，并学习更系统的评估与调参。


## 5. 小练习

自己设计一天的特征，预测营收，并写一句：你改了什么，预测如何变化。


In [ ]:
# 在这里写
